#### I have Cleaned this Datset as per requirement of Q5. But it can be used to Perform Analysis in Questions Q1.1, Q1.2, Q1.3, Q1.4, Q1.5, Q2.1–Q2.5, Q3.1, Q3.3, Q3.4, Q3.5, Q4.1, Q4.3 too.

## Introductory Idea about Cleaning
Cleans the five annual PSI-IPC CSV files into one consistent master table.
All cleaning decisions are documented inline.

The raw files have three recurring problems:
1. Slightly different column headers every year
2. Inconsistent state/UT naming (plus two administrative reorganisations)
3. "Total" sub-total rows mixed with real states

This script fixes all three so the data is ready for analysis.

#### 1. We want standard Column names for all.
- We read by position because the raw headers change slightly every year.

In [21]:

COL_NAMES = [
    'state',
    'murder', 'culpable_homicide', 'dowry_deaths', 'attempt_murder',
    'kidnapping', 'rape', 'assault_women', 'total_human_body',
    'riots', 'thefts', 'extortion', 'robbery', 'dacoity', 'prep_dacoity',
    'criminal_breach', 'cheating', 'arson', 'burglary', 'total_property',
    'counterfeiting', 'cruelty_husband', 'insult_modesty', 'total_women',
    'convicts_women', 'other_ipc', 'total_convicts'
]

### 2. State name normalisation
 Raw names vary in casing, abbreviations and punctuation.
 We also handle two structural changes:
   - J&K bifurcation (2019 data still shows one row)
   - D&N Haveli + Daman & Diu merger (Jan 2020) → we sum the 2019 rows

In [22]:
STATE_NAME_MAP = {
    'A & N ISLANDS': 'Andaman and Nicobar Islands',
    'ANDAMAN AND NICOBAR ISLANDS': 'Andaman and Nicobar Islands',
    'JAMMU & KASHMIR': 'Jammu and Kashmir',
    'JAMMU AND KASHMIR': 'Jammu and Kashmir',
    'LADAKH': 'Ladakh',
    'D & N HAVELI': 'Dadra and Nagar Haveli and Daman and Diu',
    'DAMAN & DIU': 'MERGE_INTO_PREV',          # special flag, handled below
    'DADRA AND NAGAR HAVELI AND DAMAN AND DIU': 'Dadra and Nagar Haveli and Daman and Diu',
}

In [23]:
DROP_PATTERNS = ['TOTAL', 'Total']


def load_year(path: str, year: int) -> pd.DataFrame:
    """Load one year's CSV and return a clean DataFrame."""
    raw = pd.read_csv(path, header=0)

    # Keep only the useful columns (drop serial number / Category column)
    df = raw.iloc[:, 1:28].copy()
    df.columns = COL_NAMES

    # Remove "Total (States)", "Total (All-India)" etc.
    is_agg = df['state'].str.contains('|'.join(DROP_PATTERNS), na=False)
    df = df[~is_agg].copy()

    # Uppercase for reliable matching
    df['state_upper'] = df['state'].str.strip().str.upper()

    # ── Special 2019 merge: D&N Haveli + Daman & Diu
    if year == 2019:
        numeric_cols = COL_NAMES[1:]
        dn_mask = df['state_upper'] == 'D & N HAVELI'
        dd_mask = df['state_upper'] == 'DAMAN & DIU'

        if dn_mask.any() and dd_mask.any():
            dn_idx = df.index[dn_mask][0]
            dn_vals = df.loc[dn_mask, numeric_cols].values[0]
            dd_vals = df.loc[dd_mask, numeric_cols].values[0]
            df.loc[dn_idx, numeric_cols] = dn_vals + dd_vals
            df = df[~dd_mask].copy()          # drop the second row

    # Apply name map (or Title Case anything we didn't map)
    def normalise(name_upper):
        mapped = STATE_NAME_MAP.get(name_upper)
        if mapped and mapped != 'MERGE_INTO_PREV':
            return mapped
        return name_upper.title()

    df['state'] = df['state_upper'].apply(normalise)
    df = df.drop(columns='state_upper')

    # Add year and ensure numeric types
    df['year'] = year
    df = df.reset_index(drop=True)

    for col in COL_NAMES[1:]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

In [26]:
# ── Main execution
FILES = {
    2019: '../data/raw/PSI_IPC_2019.csv',
    2020: '../data/raw/PSI_IPC_2020.csv',
    2021: '../data/raw/PSI_IPC_2021.csv',
    2022: '../data/raw/PSI_IPC_2022.csv',
    2023: '../data/raw/PSI_IPC_2023.csv',
}

yearly = []
for yr, path in FILES.items():
    df_yr = load_year(path, yr)
    yearly.append(df_yr)
    print(f"{yr}: {len(df_yr)} states/UTs loaded")

df_master = pd.concat(yearly, ignore_index=True)

print(f"\nMaster table: {df_master.shape[0]} rows × {df_master.shape[1]} columns")
print(f"Years: {sorted(df_master['year'].unique())}")
print(f"Unique states/UTs: {df_master['state'].nunique()}")


# ── Verification (kept exactly as before, just cleaner output)
print("\n── Rows per year ──")
print(df_master.groupby('year')['state'].count())

print("\n── States present in each year ──")
for s in sorted(df_master['state'].unique()):
    years = sorted(df_master[df_master['state'] == s]['year'])
    note = "  ← missing some years" if len(years) < 5 else ""
    print(f"  {s:48} {years}{note}")

print("\n── Data quality checks ──")
nan = df_master[COL_NAMES[1:]].isna().sum()
print("NaNs:", "None ✓" if nan.sum() == 0 else nan[nan > 0])

neg = {c: df_master[df_master[c] < 0][['state', 'year', c]].values
       for c in COL_NAMES[1:] if (df_master[c] < 0).any()}
print("Negatives:", "None ✓" if not neg else neg)

# Check that total_human_body = sum of its components
computed = df_master[['murder', 'culpable_homicide', 'dowry_deaths',
                      'attempt_murder', 'kidnapping', 'rape', 'assault_women']].sum(axis=1)
mismatch = df_master[computed != df_master['total_human_body']]
print("Col 10 formula:", "Consistent ✓" if mismatch.empty else f"Mismatches: {len(mismatch)}")

# 2019 merged UT vs 2020 official value
dn19 = df_master[(df_master.state == 'Dadra and Nagar Haveli and Daman and Diu') & (df_master.year == 2019)]['total_convicts'].values[0]
dn20 = df_master[(df_master.state == 'Dadra and Nagar Haveli and Daman and Diu') & (df_master.year == 2020)]['total_convicts'].values[0]
print(f"D&N Haveli 2019 (merged): {dn19:,}   2020: {dn20:,}")

# J&K / Ladakh situation
print("\n── J&K / Ladakh rows ──")
print(df_master[df_master.state.isin(['Jammu and Kashmir', 'Ladakh'])][['state', 'year', 'total_convicts']]
      .sort_values(['state', 'year']).to_string(index=False))

# Save
out_path = '../data/processed/Cleaned_dataset_for_Q5.csv'
df_master.to_csv(out_path, index=False)
print(f"\n✓ Saved clean master → {out_path}")

2019: 35 states/UTs loaded
2020: 36 states/UTs loaded
2021: 36 states/UTs loaded
2022: 36 states/UTs loaded
2023: 36 states/UTs loaded

Master table: 179 rows × 28 columns
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Unique states/UTs: 36

── Rows per year ──
year
2019    35
2020    36
2021    36
2022    36
2023    36
Name: state, dtype: int64

── States present in each year ──
  Andaman and Nicobar Islands                      [2019, 2020, 2021, 2022, 2023]
  Andhra Pradesh                                   [2019, 2020, 2021, 2022, 2023]
  Arunachal Pradesh                                [2019, 2020, 2021, 2022, 2023]
  Assam                                            [2019, 2020, 2021, 2022, 2023]
  Bihar                                            [2019, 2020, 2021, 2022, 2023]
  Chandigarh                                       [2019, 2020, 2021, 2022, 2023]
  Chhattisgarh                                     [2019, 2020, 2021, 2022, 2023]
  

In [29]:
df2 = pd.read_csv('C:/Users/Anmol Sriwastava/OneDrive/Desktop/crime-to-conviction-analysis/data/processed/Cleaned_dataset_for_Q5.csv')

In [30]:
df2.shape

(179, 28)

In [34]:
df2.head(5)

,state,murder,culpable_homicide,dowry_deaths,attempt_murder,kidnapping,rape,assault_women,total_human_body,riots,...,burglary,total_property,counterfeiting,cruelty_husband,insult_modesty,total_women,convicts_women,other_ipc,total_convicts,year
0,Andhra Pradesh,1771,58,18,43,46,123,21,2080,0,...,13,190,2,40,0,40,202,134,2446,2019
1,Arunachal Pradesh,22,24,0,0,12,26,4,88,0,...,6,26,0,0,0,0,30,19,133,2019
2,Assam,1914,187,50,118,27,246,8,2550,2,...,0,219,2,8,1,9,313,75,2857,2019
3,Bihar,3907,273,570,913,347,345,28,6383,14,...,18,853,10,110,9,119,1062,84,7463,2019
4,Chhattisgarh,5380,232,64,294,159,933,29,7091,6,...,21,275,3,14,2,16,1042,222,7613,2019


In [35]:
df2.tail(5)

,state,murder,culpable_homicide,dowry_deaths,attempt_murder,kidnapping,rape,assault_women,total_human_body,riots,...,burglary,total_property,counterfeiting,cruelty_husband,insult_modesty,total_women,convicts_women,other_ipc,total_convicts,year
174,Delhi,658,18,15,141,52,426,10,1320,9,...,0,391,3,0,38,38,489,87,1848,2023
175,Jammu and Kashmir,79,1,0,1,0,35,1,117,0,...,1,6,0,0,0,0,36,47,170,2023
176,Ladakh,1,0,0,0,0,1,0,2,0,...,0,1,0,0,0,0,1,0,3,2023
177,Lakshadweep,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2023
178,Puducherry,45,3,0,0,5,4,0,57,0,...,0,20,0,0,0,0,4,8,85,2023
